# Projeto IA – Previsão do vencedor do Mundial 2026

**Unidade Curricular:** Inteligência Artificial  
**Tema:** Análise de dados das World Cups e rankings FIFA para previsão do Mundial 2026  

**Autores:** \<António Ferreira, Mafalda Barão, Gonsalo Gomes, Rúben Dias, João Morais>  
**Docente:** \<Rui Fernandes>  
**Data:** \<11/2025>

---

## Objetivo do notebook

Neste notebook vamos:

1. Carregar e explorar 3 datasets:
   - `WorldCups.csv` – informação agregada por edição do Mundial.
   - `WorldCupMatches.csv` – informação ao nível de cada jogo.
   - `fifa_ranking-2024-06-20.csv` – histórico de rankings FIFA das seleções.
2. Fazer análise exploratória (EDA) dos dados.
3. Integrar informação dos rankings FIFA com as edições do Mundial.
4. Construir um primeiro modelo simples para prever o vencedor de uma edição do Mundial.
5. Discutir como isto pode ser usado (mais tarde) para estimar o vencedor do Mundial 2026.


# Imports principais

In [ ]:
# Bibliotecas principais
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Modelos de ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Descrição dos datasets

## 1. Descrição dos datasets

- **WorldCups.csv**
  - Uma linha por edição da World Cup.
  - Colunas principais: `Year`, `Country` (país anfitrião), `Winner`, `Runners-Up`,
    `GoalsScored`, `QualifiedTeams`, `MatchesPlayed`, `Attendance`, etc.

- **WorldCupMatches.csv**
  - Uma linha por jogo.
  - Colunas principais: `Year`, `Stage`, `Home Team Name`, `Home Team Goals`,
    `Away Team Name`, `Away Team Goals`, `Attendance`, etc.

- **fifa_ranking-2024-06-20.csv**
  - Várias linhas por seleção ao longo do tempo.
  - Colunas principais: `country_full`, `rank`, `total_points`, `confederation`,
    `rank_date`, etc.



# Carregar datasets

In [ ]:
# Carregar os datasets (ajusta o caminho se necessário)
wc = pd.read_csv("WorldCups.csv")
matches = pd.read_csv("WorldCupMatches.csv")
ranking = pd.read_csv("fifa_ranking-2024-06-20.csv")

print("WorldCups:")
display(wc.head())

print("\nWorldCupMatches:")
display(matches.head())

print("\nFIFA Ranking:")
display(ranking.head())

# Limpeza básica e tipos

In [ ]:
# Ver informação geral
print("WorldCups info:")
wc.info()
print("\nWorldCupMatches info:")
matches.info()
print("\nRanking info:")
ranking.info()

# --- Limpeza WorldCups ---

# Tirar espaços dos nomes das colunas e substituir espaços por underscore
wc.columns = [c.strip().replace(" ", "_") for c in wc.columns]

# Converter Year para inteiro (garantia)
wc["Year"] = wc["Year"].astype(int)

# Attendance vem como string com pontos como separador de milhares (ex: "3.587.538")
if "Attendance" in wc.columns:
    wc["Attendance"] = wc["Attendance"].astype(str).str.replace(".", "", regex=False)
    wc["Attendance"] = pd.to_numeric(wc["Attendance"], errors="coerce")

display(wc.head())

# --- Limpeza WorldCupMatches ---

matches.columns = [c.strip().replace(" ", "_") for c in matches.columns]
matches["Year"] = matches["Year"].astype(int)

# Goals e Attendance para numérico
for col in ["Home_Team_Goals", "Away_Team_Goals", "Half-time_Home_Goals", "Half-time_Away_Goals"]:
    if col in matches.columns:
        matches[col] = pd.to_numeric(matches[col], errors="coerce")

if "Attendance" in matches.columns:
    matches["Attendance"] = matches["Attendance"].astype(str).str.replace(".", "", regex=False)
    matches["Attendance"] = pd.to_numeric(matches["Attendance"], errors="coerce")

display(matches.head())

# --- Limpeza Ranking FIFA ---

ranking["rank_date"] = pd.to_datetime(ranking["rank_date"])
ranking["year"] = ranking["rank_date"].dt.year

display(ranking.head())


# EDA: WorldCups

In [ ]:
# Distribuição de títulos por seleção
if "Winner" in wc.columns:
    winners_counts = wc["Winner"].value_counts()
    sns.barplot(x=winners_counts.index, y=winners_counts.values)
    plt.title("Número de títulos por seleção")
    plt.xticks(rotation=45)
    plt.ylabel("N.º de títulos")
    plt.show()

# Golos por edição
if "GoalsScored" in wc.columns:
    sns.lineplot(data=wc, x="Year", y="GoalsScored", marker="o")
    plt.title("Golos marcados por edição")
    plt.show()

# Evolução da assistência total
if "Attendance" in wc.columns:
    sns.lineplot(data=wc, x="Year", y="Attendance", marker="o")
    plt.title("Assistência total por edição")
    plt.show()

# Número de equipas e jogos
if {"QualifiedTeams", "MatchesPlayed"}.issubset(wc.columns):
    fig, ax = plt.subplots()
    sns.lineplot(data=wc, x="Year", y="QualifiedTeams", marker="o", ax=ax, label="Equipas")
    sns.lineplot(data=wc, x="Year", y="MatchesPlayed", marker="o", ax=ax, label="Jogos")
    plt.title("Equipas qualificadas e jogos por edição")
    plt.legend()
    plt.show()


# EDA: WorldCupMatches

In [ ]:
# Total de golos por jogo
matches["Total_Goals"] = matches["Home_Team_Goals"] + matches["Away_Team_Goals"]

sns.histplot(matches["Total_Goals"].dropna(), bins=10)
plt.title("Distribuição de golos por jogo")
plt.xlabel("Golos no jogo")
plt.show()

# Média de golos por edição
avg_goals_year = matches.groupby("Year")["Total_Goals"].mean().reset_index()
sns.lineplot(data=avg_goals_year, x="Year", y="Total_Goals", marker="o")
plt.title("Média de golos por jogo em cada edição")
plt.ylabel("Média de golos por jogo")
plt.show()

# Distribuição dos jogos por fase (Stage)
if "Stage" in matches.columns:
    stage_counts = matches["Stage"].value_counts()
    sns.barplot(y=stage_counts.index, x=stage_counts.values)
    plt.title("Número de jogos por fase")
    plt.xlabel("N.º de jogos")
    plt.ylabel("Fase")
    plt.show()


# EDA: Ranking FIFA + exemplo com Portugal

In [ ]:
# Exemplo: evolução do ranking de Portugal ao longo do tempo
portugal_rank = ranking[ranking["country_full"] == "Portugal"].sort_values("rank_date")

plt.plot(portugal_rank["rank_date"], portugal_rank["rank"])
plt.gca().invert_yaxis()  # rank 1 é melhor => fica em cima
plt.title("Evolução do ranking FIFA – Portugal")
plt.xlabel("Data")
plt.ylabel("Posição no ranking (1 = melhor)")
plt.show()

# Número de países por confederação ao longo do tempo (última data de cada ano)
last_rank_per_year = ranking.sort_values("rank_date").drop_duplicates(
    subset=["year", "country_full"], keep="last"
)

conf_counts = last_rank_per_year.groupby(["year", "confederation"])["country_full"].nunique().reset_index()
sns.lineplot(data=conf_counts, x="year", y="country_full", hue="confederation", marker="o")
plt.title("N.º de países por confederação ao longo dos anos (ranking FIFA)")
plt.ylabel("N.º de países")
plt.show()
